# The Honest Friend — Dataset Exploration

Exploratory analysis of the Yelp Academic Dataset used for persona modelling.

**Dataset:** 500 users × 15,209 reviews  
**Purpose:** Understand rating distributions, review verbosity, and user behaviour patterns that inform persona extraction design.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('../data/sample/reviews_sample.csv')
print(f'Total reviews: {len(df)}')
print(f'Unique users: {df["user_id"].nunique()}')
print(f'Avg reviews per user: {len(df) / df["user_id"].nunique():.1f}')
df.head()

## 1. Rating Distribution

Understanding the global rating distribution helps set the naive baseline (mean-prediction RMSE).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df['stars'].value_counts().sort_index().plot(kind='bar', ax=ax, color='#C9A84C')
ax.set_title('Rating Distribution (500 users, 15,209 reviews)')
ax.set_xlabel('Star Rating')
ax.set_ylabel('Count')
ax.set_facecolor('#0D1B2A')
fig.patch.set_facecolor('#0D1B2A')
ax.tick_params(colors='white')
ax.title.set_color('white')
ax.xaxis.label.set_color('white')
ax.yaxis.label.set_color('white')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(f'Global mean rating: {df["stars"].mean():.2f}')
print(f'Global std: {df["stars"].std():.2f}')
print(f'Naive baseline RMSE (always predict mean): {np.sqrt(((df["stars"] - df["stars"].mean())**2).mean()):.4f}')

## 2. Per-User Rating Style Distribution

The PersonaBuilder classifies users as generous (avg ≥ 4.2), critical (avg ≤ 2.5), or balanced.  
This cell shows the distribution across the 500-user sample.

In [ ]:
user_avg = df.groupby('user_id')['stars'].mean()

def rating_style(avg):
    if avg >= 4.2:
        return 'generous'
    elif avg <= 2.5:
        return 'critical'
    return 'balanced'

styles = user_avg.apply(rating_style).value_counts()
print('Rating style distribution across 500 users:')
print(styles)
print(f'\nMost users are balanced — this is why persona extraction matters for the minority.')

## 3. Review Verbosity

Verbosity (brief / moderate / detailed) is a key persona signal.  
Cutoffs: brief < 60 words, detailed > 150 words.

In [ ]:
df['word_count'] = df['text'].str.split().str.len()

print(f'Median review length: {df["word_count"].median():.0f} words')
print(f'Mean review length:   {df["word_count"].mean():.0f} words')
print(f'Max review length:    {df["word_count"].max()} words')

brief    = (df['word_count'] < 60).sum()
detailed = (df['word_count'] > 150).sum()
moderate = len(df) - brief - detailed

print(f'\nVerbosity breakdown:')
print(f'  Brief    (<60 words):  {brief:,} reviews ({brief/len(df)*100:.1f}%)')
print(f'  Moderate (60-150):     {moderate:,} reviews ({moderate/len(df)*100:.1f}%)')
print(f'  Detailed (>150 words): {detailed:,} reviews ({detailed/len(df)*100:.1f}%)')

## 4. User Consistency (Rating Standard Deviation)

Consistency (low std = consistent rater) is extracted as a persona signal.  
High-consistency users are more predictable — the agent can rely more heavily on their pattern.

In [ ]:
user_std = df.groupby('user_id')['stars'].std().dropna()

print(f'Median user rating std: {user_std.median():.2f}')
print(f'Users with std < 1.0 (consistent): {(user_std < 1.0).sum()} / {len(user_std)}')
print(f'Users with std > 1.5 (volatile):   {(user_std > 1.5).sum()} / {len(user_std)}')

## 5. Evaluation Results Summary

Results from `core/evaluate.py` — holdout evaluation on 20 users.

In [ ]:
import json

with open('../paper/eval_results.json') as f:
    results = json.load(f)

print('=== Task A Evaluation Results ===')
print(f'Users evaluated : {results["n_users"]}')
print(f'RMSE            : {results["rmse"]}  (baseline: 1.0193, improvement: 24.3%)')
print(f'Avg ROUGE-L     : {results["rouge_l"]}')
print(f'Avg rating delta: {results["avg_delta"]} stars')